# A vectorized 2-layer MLP in numpy, with a hand-written backward pass

This notebook implements, with numpy only:

```text
X (N,D) --W1,b1--> Z1 (N,H) --ReLU--> A1 (N,H) --W2,b2--> Z2 (N,C) --softmax+CE--> loss
```

It matches these sections of notes.md:

- Section 2 (Backpropagation): the matrix backward rules $\partial L/\partial W = X^\top G$ and $\partial L/\partial X = G W^\top$.
- Section 3 (Losses): softmax cross-entropy, its gradient $p - y$, and the log-sum-exp trick (`stable_logsumexp`).
- Section 5 (Initialization): Kaiming init for the ReLU layer.
- Section 6 (Optimization): SGD, momentum and Adam with bias correction.

The demos, in order: a numerically stable softmax cross-entropy, a forward pass, a gradient check of the hand-written backward pass, a linear baseline, two small optimizer checks, and a training comparison of SGD, momentum and Adam on a 3-class spiral. Everything runs on CPU in under a second. Run it with
`uv run jupyter execute ml/03_neural_networks/mlp_numpy.ipynb`.

In [1]:
import numpy as np

## Data: a 3-class spiral

Each class is one arm of a spiral in 2-D (the CS231n example), with 100 points per class. The arms wind around each other, so no straight line separates any class from the others; a linear classifier cannot fit this data well, and a hidden layer is needed. The next cell generates the data and prints its shape and class counts.

In [2]:
def make_spiral(points_per_class=100, n_classes=3, noise=0.2):
    N, C = points_per_class, n_classes
    X = np.zeros((N * C, 2))
    y = np.zeros(N * C, dtype=int)
    for c in range(C):
        ix = range(N * c, N * (c + 1))
        r = np.linspace(0.0, 1.0, N)                                         # radius
        t = np.linspace(c * 4, (c + 1) * 4, N) + np.random.randn(N) * noise  # angle
        X[ix] = np.c_[r * np.sin(t), r * np.cos(t)]
        y[ix] = c
    return X, y


np.random.seed(42)
X_spiral, y_spiral = make_spiral(points_per_class=100, n_classes=3)
print("X shape:", X_spiral.shape, " class counts:", np.bincount(y_spiral))

X shape: (300, 2)  class counts: [100 100 100]


## Numerically stable softmax cross-entropy

Softmax is unchanged when the same constant $m$ is subtracted from every logit, because $e^{-m}$ cancels between numerator and denominator. Choosing $m = \max_j z_j$ makes every exponent $\le 0$ (no overflow) and makes one term equal to 1 (so the sum is $\ge 1$ and its log is finite). The loss is computed directly as a log-probability, $\log p_i = z_i - \mathrm{logsumexp}(z)$, never as `log(softmax(z))`, because the log of a probability that underflowed to 0.0 is `-inf`. The gradient of the mean loss with respect to the logits is $(p - y_{\text{onehot}})/N$ (notes section 3).

The next cell defines `stable_logsumexp` and `softmax_ce`, then runs them on the logits $z = (1000, 1001, 1002)$ from Exercise 5. Expected: the naive formula returns `nan` (because $e^{1000}$ overflows to `inf` and `inf/inf = nan`), while the stable one returns $p \approx (0.0900, 0.2447, 0.6652)$; each row of the gradient sums to 0.

In [3]:
def stable_logsumexp(Z):
    # log(sum_j exp(z_j)) per row, shifted by the row max so no exponent is positive
    m = Z.max(axis=1, keepdims=True)
    return m + np.log(np.exp(Z - m).sum(axis=1, keepdims=True))


def softmax_ce(Z, y):
    # Return (mean loss, dLoss/dZ) for logits Z (N,C) and integer labels y (N,).
    # Per example: L_i = -log p_{i,y_i} = -z_{i,y_i} + logsumexp(z_i), and
    # dL_i/dz_k = p_k - 1[k == y_i]. The mean over N adds a factor 1/N.
    logp = Z - stable_logsumexp(Z)                # (N,C) log-probabilities
    N = Z.shape[0]
    loss = -logp[np.arange(N), y].mean()
    P = np.exp(logp)                              # softmax probabilities
    dZ = P.copy()
    dZ[np.arange(N), y] -= 1.0                    # p - y  (y as one-hot)
    dZ /= N
    return loss, dZ


z = np.array([[1000.0, 1001.0, 1002.0]])
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(z) / np.exp(z).sum()
print("naive softmax :", naive)
print("stable softmax:", np.exp(z - stable_logsumexp(z)).round(4))
loss, dZ = softmax_ce(z, np.array([2]))
print(f"CE loss for true class 2: {loss:.4f}  (= -log 0.6652)")
print("dLoss/dz = p - y:", dZ.round(4), f" row sum: {abs(dZ.sum()):.1e}")
assert np.isnan(naive).all() and abs(dZ.sum()) < 1e-12

naive softmax : [[nan nan nan]]
stable softmax: [[0.09   0.2447 0.6652]]
CE loss for true class 2: 0.4076  (= -log 0.6652)
dLoss/dz = p - y: [[ 0.09    0.2447 -0.3348]]  row sum: 4.8e-14


## The model: forward pass and manual backward pass

For a linear layer $Z = AW + b$ with upstream gradient $G = \partial L/\partial Z$:

- $\partial L/\partial W = A^\top G$. Element-wise, $Z_{nc} = \sum_h A_{nh} W_{hc} + b_c$, so $\partial Z_{nc}/\partial W_{hc} = A_{nh}$. The weight $W_{hc}$ affects $Z_{nc}$ for every example $n$, and the chain rule sums over all of them: $\partial L/\partial W_{hc} = \sum_n A_{nh} G_{nc} = (A^\top G)_{hc}$.
- $\partial L/\partial A = G W^\top$. The entry $A_{nh}$ affects $Z_{nc}$ for every class $c$ with $\partial Z_{nc}/\partial A_{nh} = W_{hc}$, so $\partial L/\partial A_{nh} = \sum_c G_{nc} W_{hc} = (G W^\top)_{nh}$.
- $\partial L/\partial b = \sum_n G_{n,:}$, the column sum of $G$, since $b_c$ adds to every example.

The ReLU backward multiplies the gradient by the mask $Z_1 > 0$. Shapes give a quick check: $(D,H) = (D,N)(N,H)$ and $(N,H) = (N,C)(C,H)$ fit in only one arrangement of transposes. The forward pass returns a cache $(X, Z_1, A_1)$ because the backward pass needs these forward values; this is why training memory grows with the size of the activations.

The next cell defines the model and runs one forward pass on the spiral data. Expected: logits of shape (300, 3) and an initial loss near $\ln 3 \approx 1.10$, the loss of a uniform prediction over 3 classes (it is somewhat above that, since random initial logits are not exactly equal).

In [4]:
def init_params(D, H, C):
    # Kaiming init for the ReLU hidden layer (std = sqrt(2/fan_in)); std = sqrt(1/fan_in)
    # for the linear output layer. Derivations in notes.md section 5.
    return {
        "W1": np.random.randn(D, H) * np.sqrt(2.0 / D),
        "b1": np.zeros(H),
        "W2": np.random.randn(H, C) * np.sqrt(1.0 / H),
        "b2": np.zeros(C),
    }


def forward(params, X):
    Z1 = X @ params["W1"] + params["b1"]     # (N,H)  pre-activations
    A1 = np.maximum(0.0, Z1)                 # (N,H)  ReLU
    Z2 = A1 @ params["W2"] + params["b2"]    # (N,C)  logits
    return Z2, (X, Z1, A1)


def backward(params, cache, dZ2):
    X, Z1, A1 = cache
    grads = {}
    # output layer Z2 = A1 @ W2 + b2
    grads["W2"] = A1.T @ dZ2                 # (H,C) = (H,N) @ (N,C)
    grads["b2"] = dZ2.sum(axis=0)            # (C,)
    dA1 = dZ2 @ params["W2"].T               # (N,H) = (N,C) @ (C,H)
    # ReLU: gradient passes only where the unit was active
    dZ1 = dA1 * (Z1 > 0)                     # (N,H)
    # hidden layer Z1 = X @ W1 + b1 (same rule)
    grads["W1"] = X.T @ dZ1                  # (D,H) = (D,N) @ (N,H)
    grads["b1"] = dZ1.sum(axis=0)            # (H,)
    return grads


def loss_and_grads(params, X, y):
    Z2, cache = forward(params, X)
    loss, dZ2 = softmax_ce(Z2, y)
    return loss, backward(params, cache, dZ2)


np.random.seed(42)
params = init_params(D=2, H=64, C=3)
Z2, _ = forward(params, X_spiral)
loss0, _ = softmax_ce(Z2, y_spiral)
print("logits shape:", Z2.shape)
print(f"initial loss: {loss0:.4f}   (ln 3 = {np.log(3):.4f})")

logits shape: (300, 3)
initial loss: 1.1932   (ln 3 = 1.0986)


## Gradient check: hand-written backward vs finite differences

A gradient check compares each analytic gradient entry with the centered finite difference $(L(\theta + h) - L(\theta - h)) / (2h)$. Centered differences have $O(h^2)$ error, because the odd terms of the Taylor expansion cancel, versus $O(h)$ for a one-sided difference. The next cell uses a tiny problem (7 points, hidden width 5, float64) and checks every entry of every parameter tensor. Expected: a maximum relative error around $10^{-9}$ or smaller for each tensor; the cell asserts it is below $10^{-6}$.

In [5]:
def gradient_check():
    np.random.seed(42)
    X = np.random.randn(7, 2)
    y = np.random.randint(0, 3, size=7)
    params = init_params(D=2, H=5, C=3)
    _, grads = loss_and_grads(params, X, y)

    h = 1e-5
    for name, W in params.items():
        num = np.zeros_like(W)
        for idx in np.ndindex(W.shape):  # every entry (fine at this size)
            old = W[idx]
            W[idx] = old + h
            lp, _ = loss_and_grads(params, X, y)
            W[idx] = old - h
            lm, _ = loss_and_grads(params, X, y)
            W[idx] = old
            num[idx] = (lp - lm) / (2 * h)
        rel = np.abs(grads[name] - num) / (np.abs(grads[name]) + np.abs(num) + 1e-12)
        print(f"  {name}: max relative error = {rel.max():.3e}")
        assert rel.max() < 1e-6, f"gradient check FAILED for {name}"
    print("PASS: the hand-derived matrix gradients match finite differences")


gradient_check()

  W1: max relative error = 1.007e-09
  b1: max relative error = 3.827e-11
  W2: max relative error = 3.722e-10
  b2: max relative error = 9.411e-11
PASS: the hand-derived matrix gradients match finite differences


## Baseline: a linear softmax classifier on the spiral

To confirm that the hidden layer is needed, the next cell trains a linear model $Z = XW + b$ with the same softmax cross-entropy and full-batch gradient descent (1000 steps, learning rate 1.0). Expected: training accuracy close to 50%, far below what the 2-layer MLP reaches later in this notebook.

In [6]:
np.random.seed(42)
W_lin = np.random.randn(2, 3) * 0.01
b_lin = np.zeros(3)
for _ in range(1000):
    Z = X_spiral @ W_lin + b_lin
    loss_lin, dZ = softmax_ce(Z, y_spiral)
    W_lin -= 1.0 * (X_spiral.T @ dZ)   # same linear-layer rule: dW = X^T G
    b_lin -= 1.0 * dZ.sum(axis=0)
acc_lin = ((X_spiral @ W_lin + b_lin).argmax(axis=1) == y_spiral).mean()
print(f"linear model: loss {loss_lin:.4f}, training accuracy {acc_lin * 100:.1f}%")
assert acc_lin < 0.65

linear model: loss 0.7458, training accuracy 51.7%


## Optimizers, with two small checks

All three optimizers share one interface, `update(params, grads, t)`, so the comparison below changes nothing except the update rule.

- Momentum keeps $v \leftarrow \beta v + g$ and steps by $\eta v$. For a constant gradient $g$, $v$ converges to $g/(1-\beta)$, which is $10g$ at $\beta = 0.9$ (notes section 6).
- Adam keeps EMAs $m$ and $v$ of $g$ and $g^2$, starting at 0, and divides them by $1-\beta_1^t$ and $1-\beta_2^t$ (bias correction). With both corrections the first step is exactly $\eta \cdot \mathrm{sign}(g)$ (up to $\epsilon$). With $m$ corrected and $v$ not, the first step is $1/\sqrt{1-\beta_2} \approx 31.6$ times larger; with neither corrected it is $0.1/\sqrt{0.001} \approx 3.2$ times larger.

The next cell defines the optimizers and checks both claims on a single scalar parameter with gradient $g = 1$. Expected: the momentum velocity after 100 steps is $10(1 - 0.9^{100}) \approx 9.9997$, and Adam's first step is 0.0100 for $\eta = 0.01$.

In [7]:
def make_sgd(lr):
    def update(params, grads, t):
        for k in params:
            params[k] -= lr * grads[k]
    return update


def make_momentum(lr, beta=0.9):
    v = {}
    def update(params, grads, t):
        for k in params:
            v[k] = beta * v.get(k, 0.0) + grads[k]   # sum of past gradients, weights beta^age
            params[k] -= lr * v[k]
    update.velocity = v                              # exposed for the check below
    return update


def make_adam(lr, beta1=0.9, beta2=0.999, eps=1e-8):
    m, v = {}, {}
    def update(params, grads, t):  # t is 1-based
        for k in params:
            m[k] = beta1 * m.get(k, 0.0) + (1 - beta1) * grads[k]
            v[k] = beta2 * v.get(k, 0.0) + (1 - beta2) * grads[k] ** 2
            # Bias correction: E[m_t] = g * (1 - beta1^t) because m starts at 0,
            # so dividing by (1 - beta1^t) removes the bias (same for v with beta2).
            mhat = m[k] / (1 - beta1 ** t)
            vhat = v[k] / (1 - beta2 ** t)
            params[k] -= lr * mhat / (np.sqrt(vhat) + eps)
    return update


# Check 1: momentum velocity under a constant gradient approaches g / (1 - beta).
p, mom = {"w": np.array(0.0)}, make_momentum(lr=0.01, beta=0.9)
for t in range(1, 101):
    mom(p, {"w": np.array(1.0)}, t)
print(f"momentum velocity after 100 steps of g=1: {float(mom.velocity['w']):.4f}  (g/(1-beta) = 10)")

# Check 2: Adam's first step with bias correction is lr * sign(g).
p, adam = {"w": np.array(0.0)}, make_adam(lr=0.01)
adam(p, {"w": np.array(1.0)}, 1)
print(f"Adam first step for g=1, lr=0.01: {-float(p['w']):.4f}")
assert abs(float(mom.velocity["w"]) - 10.0) < 1e-3 and abs(-float(p["w"]) - 0.01) < 1e-8

momentum velocity after 100 steps of g=1: 9.9997  (g/(1-beta) = 10)
Adam first step for g=1, lr=0.01: 0.0100


## Training comparison: SGD vs momentum vs Adam

The next cell trains the 2-64-3 MLP on the spiral for 300 full-batch steps with each optimizer, from the same initial weights (the seed is reset before each run). It prints the loss every 30 steps and the final training accuracy. The learning rates are reasonable hand-picked values, not tuned for each optimizer: SGD 1.0, momentum 0.5 with $\beta = 0.9$, Adam 0.01. Expected: momentum reaches the lowest loss at every logged step after the first; Adam starts slower than SGD and ends with a lower loss; all three exceed 95% training accuracy, versus about 50% for the linear baseline.

In [8]:
def train(X, y, optimizer_factory, n_steps=300, H=64, log_every=30):
    np.random.seed(42)  # identical init for every optimizer
    params = init_params(D=X.shape[1], H=H, C=y.max() + 1)
    update = optimizer_factory()
    history = []
    for t in range(1, n_steps + 1):
        loss, grads = loss_and_grads(params, X, y)  # full batch: the dataset is tiny
        update(params, grads, t)
        if t % log_every == 0 or t == 1:
            history.append((t, loss))
    Z2, _ = forward(params, X)
    acc = (Z2.argmax(axis=1) == y).mean()
    return history, acc


runs = {
    "SGD(lr=1.0)":            lambda: make_sgd(lr=1.0),
    "Momentum(lr=0.5,b=0.9)": lambda: make_momentum(lr=0.5, beta=0.9),
    "Adam(lr=0.01)":          lambda: make_adam(lr=0.01),
}
results = {name: train(X_spiral, y_spiral, factory) for name, factory in runs.items()}

steps = [t for t, _ in results["SGD(lr=1.0)"][0]]
header = f"{'step':>6} | " + " | ".join(f"{n:>22}" for n in runs)
print(header)
print("-" * len(header))
for i, t in enumerate(steps):
    print(f"{t:>6} | " + " | ".join(f"{results[n][0][i][1]:>22.4f}" for n in runs))
print()
for name in runs:
    print(f"  final train accuracy [{name:>22}]: {results[name][1] * 100:5.1f}%")
assert results["Adam(lr=0.01)"][1] > 0.90, "Adam failed to fit the spiral"

  step |            SGD(lr=1.0) | Momentum(lr=0.5,b=0.9) |          Adam(lr=0.01)
---------------------------------------------------------------------------------
     1 |                 1.1932 |                 1.1932 |                 1.1932
    30 |                 0.5002 |                 0.2095 |                 0.5760
    60 |                 0.3107 |                 0.0946 |                 0.3335
    90 |                 0.2245 |                 0.0718 |                 0.2006
   120 |                 0.1810 |                 0.0611 |                 0.1419
   150 |                 0.1544 |                 0.0544 |                 0.1106
   180 |                 0.1363 |                 0.0496 |                 0.0914
   210 |                 0.1229 |                 0.0460 |                 0.0782
   240 |                 0.1127 |                 0.0431 |                 0.0686
   270 |                 0.1045 |                 0.0407 |                 0.0613
   300 |        

**Reading the table.** All three runs start from the same loss because the initial weights are identical. Momentum's loss is the lowest at every logged step after the first: along directions where successive gradients agree, its velocity adds them up (up to $1/(1-\beta) = 10$ times one gradient), so it moves further per step than SGD. Adam rescales each parameter's step by the root of its own squared-gradient EMA, so one global learning rate gives similar step sizes for all tensors. Its per-parameter step is at most about $\eta = 0.01$, so it starts slower than SGD (higher loss at step 30) and has the lower loss from step 90 on. A single run with untuned learning rates does not rank optimizers in general; it shows what each update rule does on one problem.